# Deep Learning Project: Teach a CNN Everyday Objects

**Group members:** *write your names here*

Fill in every `# TODO`, run every cell, and answer each question in its **Your answer** cell. See the project handout for the deliverables, the deadlines and how points are awarded.

## Setup

Run this notebook on **Google Colab** (Runtime → Change runtime type → **T4 GPU**) or on your laptop (`pip install torch torchvision scikit-learn pandas matplotlib`).

Your image folder must look like this, with the **same category names** in `train` and `test`:

```
my_objects/
    train/   mug/  bottle/  can/      (at least 50 photos each)
    test/    mug/  bottle/  can/      (at least 15 photos each, never used for training)
    hard/    mug/  bottle/  can/      (optional: your tricky photos for Part 5E)
```

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageOps, ImageFilter, ImageEnhance, ImageDraw
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("Using", device)

In [ ]:
# On Google Colab: uncomment the next two lines to read your images from Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

# TODO: set DATA_DIR to your folder (on Colab, e.g. "/content/drive/MyDrive/my_objects")

The cell below reads every image once and keeps it in memory. It turns phone photos the right way up (phones store the rotation separately) and shrinks them to at most 448 pixels, so 200 photos fit easily in memory.

We also hold out 20% of the training photos as a **validation set**. We watch it during training to see overfitting. The **test set** is used only to report final accuracy.

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".jfif", ".webp", ".bmp"}

def load_folder(folder, classes=None):
    folder = Path(folder)
    if classes is None:
        classes = sorted(d.name for d in folder.iterdir() if d.is_dir() and not d.name.startswith("."))
    imgs, labels = [], []
    for k, c in enumerate(classes):
        for f in sorted((folder / c).iterdir()):
            if f.suffix.lower() in IMG_EXT:
                im = ImageOps.exif_transpose(Image.open(f)).convert("RGB")
                im.thumbnail((448, 448))
                imgs.append(im); labels.append(k)
    return imgs, np.array(labels), classes

train_imgs, train_y, classes = load_folder(Path(DATA_DIR) / "train")
test_imgs, test_y, _ = load_folder(Path(DATA_DIR) / "test", classes)
K = len(classes)

idx_tr, idx_val = train_test_split(np.arange(len(train_y)), test_size=0.2, random_state=1, stratify=train_y)
print("Categories:", classes)
print(pd.DataFrame({"train": np.bincount(train_y[idx_tr]), "validation": np.bincount(train_y[idx_val]),
                    "test": np.bincount(test_y)}, index=classes))

## Part 1. Your data (10 points)

### Question 1

The cell shows 5 training photos per category. Then describe your data:

a. Which categories did you choose, and how many **different objects** did you photograph per category? (At least 5 different mugs, not 50 photos of one mug.)

b. How did you make the photos varied (backgrounds, angles, lighting, distance)? How is your test set different from your training set?

In [ ]:
fig, axes = plt.subplots(K, 5, figsize=(10, 2.1 * K), squeeze=False)
for row, k in enumerate(range(K)):
    for col, i in enumerate(np.where(train_y == k)[0][:5]):
        axes[row, col].imshow(train_imgs[i]); axes[row, col].axis("off")
    axes[row, 0].set_title(classes[k], loc="left")
plt.tight_layout(); plt.show()

**Your answer:** *Write your answer here.*

## Part 2. Baselines (10 points)

Every later model has to beat these. Both use the raw pixels of each photo shrunk to 64×64, i.e. 64 × 64 × 3 = 12,288 numbers per photo.

### Question 2

a. Train a `RandomForestClassifier(n_estimators=300, random_state=1)` and an `MLPClassifier(hidden_layer_sizes=(128,), max_iter=500, random_state=1)` on the training pixels. Report the test accuracy of each.

b. What accuracy would you get by always guessing the largest category? Do the baselines beat it?

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

def pixels(imgs):
    return np.stack([np.asarray(im.resize((64, 64))) / 255 for im in imgs]).reshape(len(imgs), -1)

X_train, X_test = pixels([train_imgs[i] for i in idx_tr]), pixels(test_imgs)
y_tr = train_y[idx_tr]
results = {}

# TODO: fit both models on (X_train, y_tr) and store their test accuracy in results
print("Always guessing the largest category:", round(np.bincount(test_y).max() / len(test_y), 3))
results

**Your answer:** *Write your answer here.*

## Part 3. A CNN from scratch (15 points)

The helpers below are used by every network in this project. `train_model` trains on the training photos (with **data augmentation**: random crops, flips, small rotations and color changes, so the network sees a slightly different version of each photo every epoch) and records training and validation accuracy after every epoch.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

class ImageList(Dataset):
    def __init__(self, imgs, labels, tf):
        self.imgs, self.labels, self.tf = imgs, labels, tf
    def __len__(self):
        return len(self.imgs)
    def __getitem__(self, i):
        return self.tf(self.imgs[i]), int(self.labels[i])

def predict_proba(model, imgs, tf):
    model.eval()
    with torch.no_grad():
        out = [torch.softmax(model(xb.to(device)), 1).cpu()
               for xb, _ in DataLoader(ImageList(imgs, np.zeros(len(imgs)), tf), batch_size=64)]
    return torch.cat(out).numpy()

def accuracy(model, imgs, labels, tf):
    return (predict_proba(model, imgs, tf).argmax(1) == labels).mean()

def train_model(model, train_tf, eval_tf, epochs, lr=1e-3, params=None, idx=None, images=None):
    """Train on the training photos idx (default: all of idx_tr). Returns (train acc, val acc) per epoch."""
    idx = idx_tr if idx is None else idx
    images = train_imgs if images is None else images
    imgs, labels = [images[i] for i in idx], train_y[idx]
    val_imgs, val_labels = [train_imgs[i] for i in idx_val], train_y[idx_val]
    model.to(device)
    opt = torch.optim.Adam(model.parameters() if params is None else params, lr=lr)
    loader = DataLoader(ImageList(imgs, labels, train_tf), batch_size=32, shuffle=True)
    history = []
    for epoch in range(epochs):
        model.train()
        for m in model.modules():   # frozen BatchNorm layers keep their pretrained statistics
            if isinstance(m, nn.BatchNorm2d) and not any(p.requires_grad for p in m.parameters()):
                m.eval()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            # TODO: one training step: forward pass, loss, zero the gradients, backpropagation, update
        history.append((accuracy(model, imgs, labels, eval_tf), accuracy(model, val_imgs, val_labels, eval_tf)))
    return np.array(history)

def plot_history(history, title):
    plt.plot(range(1, len(history) + 1), history[:, 0], label="training")
    plt.plot(range(1, len(history) + 1), history[:, 1], label="validation")
    plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.ylim(0, 1.02); plt.title(title); plt.legend(); plt.show()

eval_tf64 = transforms.Compose([transforms.Resize((64, 64)), transforms.ToTensor()])
train_tf64 = transforms.Compose([
    transforms.RandomResizedCrop(64, scale=(0.7, 1.0)), transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15), transforms.ColorJitter(0.3, 0.3, 0.3), transforms.ToTensor()])

### Question 3

a. Complete the training step in `train_model` above (forward pass, loss, zero the gradients, backpropagation, update), as in Assignment 10.

b. Complete the CNN below for 64×64 color photos:

- Conv 3 → 16 filters (3×3, `padding=1`), ReLU, 2×2 max pooling (64×64 → 32×32)
- Conv 16 → 32 filters (3×3, `padding=1`), ReLU, 2×2 max pooling (32×32 → 16×16)
- Conv 32 → 64 filters (3×3, `padding=1`), ReLU, 2×2 max pooling (16×16 → 8×8)
- Flatten, `Dropout(0.3)`, and an output layer with `K` neurons

Train it for 40 epochs and look at the plot. Report its test accuracy. Around which epoch does it start to overfit, and how can you tell from the plot?

c. The second cell trains the same network **without** augmentation. Compare the two plots. What does augmentation change?

In [ ]:
def make_cnn():
    torch.manual_seed(1)
    return nn.Sequential(
        # TODO: the layers described in Question 3b
    )

start = time.time()
cnn = make_cnn()
hist_cnn = train_model(cnn, train_tf64, eval_tf64, epochs=40)
results["CNN from scratch"] = accuracy(cnn, test_imgs, test_y, eval_tf64)
print(f"Test accuracy: {results['CNN from scratch']:.3f}   ({time.time() - start:.0f}s)")
plot_history(hist_cnn, "CNN from scratch, with augmentation")

In [ ]:
cnn_noaug = make_cnn()
hist_noaug = train_model(cnn_noaug, eval_tf64, eval_tf64, epochs=40)
print(f"Test accuracy without augmentation: {accuracy(cnn_noaug, test_imgs, test_y, eval_tf64):.3f}")
plot_history(hist_noaug, "CNN from scratch, no augmentation")

**Your answer:** *Write your answer here.*

## Part 4. Transfer learning (20 points)

ResNet-18 is a CNN with 11 million parameters, trained on 1.2 million ImageNet photos in 1,000 categories. We reuse it in two ways:

- **Frozen:** keep all of ResNet's layers as they are, use them to turn each photo into 512 features, and train only a new last layer.
- **Fine-tuned:** also let the last block of ResNet (`layer4`) learn, with a small learning rate, so its features adapt to your photos.

ResNet expects 224×224 photos, normalized the way it was trained.

In [ ]:
weights = models.ResNet18_Weights.IMAGENET1K_V1
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
crop224 = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224)])
eval_tf224 = transforms.Compose([crop224, transforms.ToTensor(), normalize])
train_tf224 = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)), transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2), transforms.ToTensor(), normalize])

### Question 4. Frozen ResNet

a. Replace ResNet's last layer (`backbone.fc`, which predicts the 1,000 ImageNet categories) with `nn.Identity()`, so that the network outputs its 512 features.

b. The rest of the cell trains a new last layer `nn.Linear(512, K)` on those features and puts it on top of ResNet. Report the test accuracy. How many parameters were trained?

In [ ]:
backbone = models.resnet18(weights=weights)
# TODO: make the network output its 512 features instead of 1,000 ImageNet scores
backbone.to(device).eval()

def features(imgs):
    with torch.no_grad():
        return torch.cat([backbone(xb.to(device)).cpu()
                          for xb, _ in DataLoader(ImageList(imgs, np.zeros(len(imgs)), eval_tf224), batch_size=64)])

F_train = features(train_imgs)     # features of every training photo (indexed like train_imgs)
F_test = features(test_imgs)

def train_head(F, y, steps=300):
    """Train a new last layer on fixed features (all photos at once, Adam, a little weight decay)."""
    torch.manual_seed(1)
    head = nn.Linear(512, K)
    opt = torch.optim.Adam(head.parameters(), lr=1e-2, weight_decay=1e-3)
    y = torch.tensor(y)
    for _ in range(steps):
        loss = loss_fn(head(F), y)
        opt.zero_grad(); loss.backward(); opt.step()
    return head

def with_head(head):
    """A full ResNet whose last layer is the trained head (used for predictions and Grad-CAM)."""
    model = models.resnet18(weights=weights)
    model.fc = head
    return model.to(device).eval()

head = train_head(F_train[idx_tr], train_y[idx_tr])
frozen = with_head(head)
results["ResNet-18, frozen"] = accuracy(frozen, test_imgs, test_y, eval_tf224)
print(f"Test accuracy: {results['ResNet-18, frozen']:.3f}")
print("Parameters trained:", sum(p.numel() for p in head.parameters()))

**Your answer:** *Write your answer here.*

### Question 5. Fine-tuned ResNet

a. Complete the cell: freeze every parameter of `ft`, then unfreeze the parameters of `ft.layer4`, and replace `ft.fc` with a new `nn.Linear(512, K)` (a new layer is trainable automatically).

b. Train for 8 epochs. `layer4` learns with a learning rate of 0.0001, ten times smaller than the new last layer. Why should the pretrained layers learn more slowly than the new layer?

c. Report the test accuracy and compare it with the frozen version. This can take a few minutes without a GPU.

In [ ]:
torch.manual_seed(1)
ft = models.resnet18(weights=weights)
# TODO: freeze everything, unfreeze layer4, and give ft a new last layer for K categories
print("Parameters trained:", sum(p.numel() for p in ft.parameters() if p.requires_grad))

start = time.time()
param_groups = [{"params": ft.layer4.parameters(), "lr": 1e-4}, {"params": ft.fc.parameters(), "lr": 1e-3}]
hist_ft = train_model(ft, train_tf224, eval_tf224, epochs=8, params=param_groups)
results["ResNet-18, fine-tuned"] = accuracy(ft, test_imgs, test_y, eval_tf224)
print(f"Test accuracy: {results['ResNet-18, fine-tuned']:.3f}   ({time.time() - start:.0f}s)")
plot_history(hist_ft, "ResNet-18, fine-tuned")

**Your answer:** *Write your answer here.*

### Question 6. Compare the four models

The table shows every model's test accuracy. Rank them, and explain in 3–4 sentences why the ranking came out this way. Use the words *features*, *pretrained* and *number of photos*.

In [ ]:
table = pd.DataFrame({"test accuracy": results}).sort_values("test accuracy", ascending=False)
table.round(3)

**Your answer:** *Write your answer here.*

## Part 5. Stress test (20 points)

Pick **at least two** of 5A–5E. Run the cells and answer the questions for the ones you choose; skip the others. `best` is the model the stress tests use. Change it if a different model won Question 6.

In [ ]:
best, best_tf = ft, eval_tf224
best_name = "ResNet-18, fine-tuned"

### 5A. Learning curve: how many photos do you need?

The cell trains the CNN from scratch and the frozen ResNet on 5, 10, 20 and all training photos per category, and tests each on the full test set.

**Question 7A.** Describe the two curves. How many photos per category does each model need to reach 80% accuracy (if it gets there)? What does this say about when to use transfer learning?

In [ ]:
rng = np.random.default_rng(1)
sizes = [5, 10, 20, int(np.bincount(train_y[idx_tr]).min())]
curve = []
for n in sizes:
    idx_n = np.concatenate([rng.choice(idx_tr[train_y[idx_tr] == k], n, replace=False) for k in range(K)])
    small_cnn = make_cnn()
    train_model(small_cnn, train_tf64, eval_tf64, epochs=40, idx=idx_n)
    small_frozen = with_head(train_head(F_train[idx_n], train_y[idx_n]))
    curve.append((n, accuracy(small_cnn, test_imgs, test_y, eval_tf64),
                  accuracy(small_frozen, test_imgs, test_y, eval_tf224)))
    print(f"{n:3d} photos per category: CNN {curve[-1][1]:.3f}, frozen ResNet {curve[-1][2]:.3f}")

curve = np.array(curve)
plt.plot(curve[:, 0], curve[:, 1], "o-", label="CNN from scratch")
plt.plot(curve[:, 0], curve[:, 2], "o-", label="ResNet-18, frozen")
plt.axhline(1 / K, color="gray", ls=":", label="guessing")
plt.xlabel("training photos per category"); plt.ylabel("test accuracy"); plt.ylim(0, 1.02); plt.legend(); plt.show()

**Your answer:** *Write your answer here.*

### 5B. Robustness: blur, darkness and rotation

The cell changes every test photo by increasing amounts and measures the accuracy of `best`.

**Question 7B.** Which change hurts the model most? Look at the example photos: at which level would *you* stop recognizing the objects? Is the model more or less robust than you?

In [ ]:
changes = {
    "blur (radius in pixels)": ([0, 2, 4, 8], lambda im, s: im.filter(ImageFilter.GaussianBlur(s))),
    "darkness (fraction removed)": ([0, 0.5, 0.75, 0.9], lambda im, s: ImageEnhance.Brightness(im).enhance(1 - s)),
    "rotation (degrees)": ([0, 45, 90, 180], lambda im, s: im.rotate(s)),
}
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for col, (name, (levels, change)) in enumerate(changes.items()):
    accs = [accuracy(best, [change(im, s) for im in test_imgs], test_y, best_tf) for s in levels]
    axes[0, col].plot(levels, accs, "o-"); axes[0, col].set_ylim(0, 1.02)
    axes[0, col].set_xlabel(name); axes[0, col].set_ylabel("test accuracy")
    strip = np.hstack([np.asarray(change(test_imgs[0], s).resize((160, 160))) for s in levels])
    axes[1, col].imshow(strip); axes[1, col].axis("off"); axes[1, col].set_title(f"levels {levels}", fontsize=9)
plt.tight_layout(); plt.show()

**Your answer:** *Write your answer here.*

### 5C. The shortcut trap

Suppose every photo of your first category was taken on the same red table. A model can then get high accuracy by recognizing **the table instead of the object**. We fake this by stamping a red square in the corner of every *training* photo of the first category, and train the CNN from scratch on those photos.

(We use the CNN from scratch because it has to learn everything from your few photos, so it grabs the easiest signal it can find. A pretrained ResNet already recognizes the objects themselves well, so it is much less likely to fall for a red square. Real shortcuts, such as backgrounds and lighting, can fool it too.)

**Question 7C.** a. Compare the shortcut CNN with your CNN from Part 3 on clean test photos. Then stamp *every* test photo: what does the shortcut CNN predict? b. Look at the Grad-CAM maps in Question 10b: where does the shortcut CNN look? c. How would you collect your photos to avoid this trap?

In [ ]:
def stamp(im):
    im = im.copy()
    s = im.width // 4
    ImageDraw.Draw(im).rectangle([im.width - s, im.height - s, im.width, im.height], fill=(220, 30, 30))
    return im

stamped_train = [stamp(im) if y == 0 else im for im, y in zip(train_imgs, train_y)]
shortcut = make_cnn()
train_model(shortcut, train_tf64, eval_tf64, epochs=40, images=stamped_train)

stamped_test = [stamp(im) for im in test_imgs]
stamped_pred = predict_proba(shortcut, stamped_test, eval_tf64).argmax(1)
print(f"CNN from Part 3, clean test photos:   accuracy {results['CNN from scratch']:.3f}")
print(f"Shortcut CNN, clean test photos:      accuracy {accuracy(shortcut, test_imgs, test_y, eval_tf64):.3f}")
print(f"Shortcut CNN, every photo stamped:    accuracy {(stamped_pred == test_y).mean():.3f}")
print(f"Share predicted as '{classes[0]}' when stamped: {(stamped_pred == 0).mean():.3f}")

**Your answer:** *Write your answer here.*

### 5D. Frozen vs. fine-tuned

**Question 7D.** Using your results from Questions 4–5 and the training plot of the fine-tuned model: did fine-tuning help? Look at the gap between training and validation accuracy. With very few photos (say 5 per category), would you fine-tune or keep ResNet frozen? Why?

**Your answer:** *Write your answer here.*

### 5E. Your hardest photos

Take 3–5 deliberately tricky photos per category (an unusual angle, the object half out of frame or partly hidden, a cluttered background, two objects in one photo) and put them in a `hard` folder with the same subfolders as `train`.

**Question 7E.** How many does the model get right? Pick two mistakes and explain them.

In [ ]:
hard_dir = Path(DATA_DIR) / "hard"
if hard_dir.exists():
    hard_imgs, hard_y, _ = load_folder(hard_dir, classes)
    proba = predict_proba(best, hard_imgs, best_tf)
    print(f"Accuracy on hard photos: {(proba.argmax(1) == hard_y).mean():.3f} ({len(hard_y)} photos)")
    cols = 5
    fig, axes = plt.subplots(int(np.ceil(len(hard_imgs) / cols)), cols, figsize=(13, 3 * np.ceil(len(hard_imgs) / cols)), squeeze=False)
    for ax in axes.flat: ax.axis("off")
    for ax, im, y, p in zip(axes.flat, hard_imgs, hard_y, proba):
        ax.imshow(im); ax.set_title(f"true {classes[y]}\npred {classes[p.argmax()]} ({p.max():.0%})",
                                    color="black" if p.argmax() == y else "red", fontsize=9)
    plt.tight_layout(); plt.show()
else:
    print("No hard folder: skipping 5E.")

**Your answer:** *Write your answer here.*

## Part 6. Explain the model (15 points)

### Question 8. Confusion matrix

Compute the predictions of `best` on the test set and show the confusion matrix. Which two categories does it confuse most?

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

# TODO: predicted probabilities and predicted labels of best on the test photos
ConfusionMatrixDisplay.from_predictions(test_y, test_pred, display_labels=classes, colorbar=False)
plt.title(best_name); plt.show()

**Your answer:** *Write your answer here.*

### Question 9. The worst mistakes

The cell shows the 6 test photos to which the model gives the **lowest probability for the true category**: its mistakes, and the photos it nearly got wrong. For each one, give a short reason why it was hard.

In [ ]:
worst = np.argsort(test_proba[np.arange(len(test_y)), test_y])[:6]
fig, axes = plt.subplots(1, 6, figsize=(16, 3.2))
for ax, i in zip(axes, worst):
    ax.imshow(test_imgs[i]); ax.axis("off")
    ax.set_title(f"true {classes[test_y[i]]}\npred {classes[test_pred[i]]} ({test_proba[i].max():.0%})",
                 color="red" if test_pred[i] != test_y[i] else "black", fontsize=9)
plt.show()

**Your answer:** *Write your answer here.*

### Question 10. Where does the model look? (Grad-CAM)

**Grad-CAM** colors the parts of a photo that pushed the model toward its prediction. It takes the feature maps of the last convolutional layer (for ResNet, `layer4` with 7×7 positions), weights each map by how much it raised the predicted category's score (its average gradient), and adds them up. Red means "this region mattered".

a. Run the cell for 4 test photos, one of them a mistake if the model made any. Does the model look at the object or the background?

b. If you did 5C: the second row shows the shortcut CNN on stamped photos of other categories. Explain what you see.

In [ ]:
def grad_cam(model, img, layer=None, tf=None):
    """Grad-CAM heat map (224×224, values 0–1) for the model's predicted category.
    Default: a ResNet (layer4, 224×224 input). For the CNN from scratch: layer=cnn[6], tf=eval_tf64."""
    layer = model.layer4 if layer is None else layer
    tf = eval_tf224 if tf is None else tf
    model.eval()
    saved = {}
    h1 = layer.register_forward_hook(lambda m, i, out: saved.update(acts=out))
    h2 = layer.register_full_backward_hook(lambda m, gin, gout: saved.update(grads=gout[0]))
    x = tf(img).unsqueeze(0).to(device).requires_grad_(True)
    scores = model(x)
    pred = scores.argmax(1).item()
    model.zero_grad()
    scores[0, pred].backward()
    h1.remove(); h2.remove()
    w = saved["grads"].mean(dim=(2, 3), keepdim=True)              # one weight per feature map
    cam = torch.relu((w * saved["acts"]).sum(1))[0].detach().cpu().numpy()
    cam = cam / (cam.max() + 1e-8)
    return np.asarray(Image.fromarray(np.uint8(255 * cam)).resize((224, 224), Image.BILINEAR)) / 255, pred

def show_cams(model, imgs, labels, row_title, layer=None, tf=None, view=crop224):
    fig, axes = plt.subplots(1, len(imgs), figsize=(3.4 * len(imgs), 3.4), squeeze=False)
    for ax, im, y in zip(axes[0], imgs, labels):
        cam, pred = grad_cam(model, im, layer, tf)
        ax.imshow(view(im)); ax.imshow(cam, cmap="jet", alpha=0.45); ax.axis("off")
        ax.set_title(f"true {classes[y]}, pred {classes[pred]}", fontsize=9)
    fig.suptitle(row_title); plt.show()

mistakes = list(np.where(test_pred != test_y)[0][:1])
pick = mistakes + [i for i in np.argsort(-test_proba.max(1)) if i not in mistakes][:4 - len(mistakes)]
show_cams(best, [test_imgs[i] for i in pick], test_y[pick], best_name)

if "shortcut" in globals():
    other = np.where(test_y != 0)[0][::7][:4]
    show_cams(shortcut, [stamped_test[i] for i in other], test_y[other],
              "Shortcut CNN (5C) on stamped photos of other categories",
              layer=shortcut[6], tf=eval_tf64, view=lambda im: im.resize((224, 224)))

**Your answer:** *Write your answer here.*

## Part 7. Presentation (10 points)

Record a 5-minute video (every group member speaks) that covers:

1. Your categories and how you collected the photos, with a few examples
2. The table from Question 6
3. Your two stress tests and what they showed
4. The most surprising mistake, with its Grad-CAM map
5. One sentence: would you trust your model in a real app? Under what conditions?

## AI statement

You may use AI tools. Write a short statement of how you used them and anything they got wrong that you had to fix.

**Your answer:** *Write your answer here.*